In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point
from folium.plugins import TimestampedGeoJson, Timeline, TimelineSlider
from folium.features import GeoJson
from scipy.optimize import linear_sum_assignment
from folium.utilities import JsCode
import folium
import plotly.express as px
from itables import show
import itables
import json

itables.init_notebook_mode()

In [ ]:
df_raw = pd.read_csv(
    "data/raw/ratsastie20250813_0706.csv", 
    names=["topic", "payload", "zero", "boolean", "timestamp", "time_delta_prev"],
    header=None,
)
show(df_raw)

In [ ]:
df_raw.info()

In [ ]:
def parse_tracking_payload(payload: str) -> Point:
    try:
        data = json.loads(payload)
    except (TypeError, json.JSONDecodeError):
        return []
    return data.get("objects", [])
    

In [ ]:
def parse_raw_gdf(raw_data: pd.DataFrame) -> pd.DataFrame:
    raw_data = raw_data.dropna(subset=["topic"])
    tracking_mask = raw_data["topic"].str.endswith("tracking")
    tracking_df = raw_data[tracking_mask].copy()

    tracking_df["parsed_payload"] = tracking_df["payload"].apply(parse_tracking_payload)
    tracking_df = tracking_df[tracking_df["parsed_payload"].map(len) > 0]

    tracking_df = tracking_df.explode("parsed_payload", ignore_index=True)
    payload_df = pd.json_normalize(tracking_df["parsed_payload"])

    final_df = pd.concat([
        tracking_df[["timestamp", "time_delta_prev"]].reset_index(drop=True),
        payload_df,
    ], axis=1)

    final_df = final_df.drop(columns=final_df.filter(like="bounding_box").columns)
    final_df = final_df.drop(columns=final_df.filter(like="collision").columns)
    final_df = final_df.drop(columns=final_df.filter(like="acceleration").columns)
    final_df = final_df.drop(columns=final_df.filter(like="mass").columns)
    final_df = final_df.drop(columns=final_df.filter(like="previous").columns)
    final_df = final_df.drop(columns=final_df.filter(like="is_active").columns)
    final_df.rename(lambda x: x.replace(".", "_"), axis=1, inplace=True)
    final_df.sort_values(by="timestamp", inplace=True)
    final_df["time_index"] = final_df.groupby("timestamp").ngroup()

    return final_df


parsed_df = parse_raw_gdf(df_raw)

In [ ]:
show(parsed_df)

In [ ]:
parsed_df.timestamp.diff().mean()

In [ ]:
parsed_df.info()

In [ ]:
# class Track:
#     def __init__(self, x0, t0, row_idx, track_id):
#         """
#         x0: initial position (x, y)
#         t0: initial time (float)
#         row_idx: index in the sorted dataframe for this detection
#         track_id: internal sequential ID
#         """
#         self.id = track_id
        
#         # State: [x, y, vx, vy]
#         self.x = np.array([x0[0], x0[1], 0.0, 0.0], dtype=float)
#         self.P = np.eye(4) * 10.0  # large initial uncertainty
        
#         self.first_time = t0
#         self.last_time = t0
        
#         self.hits = 1
#         self.confirmed = False
        
#         # Keep which rows in the df belong to this track
#         self.row_indices = [row_idx]

#     def predict(self, t_now, process_noise_pos=1.0, process_noise_vel=1.0):
#         dt = t_now - self.last_time
#         if dt <= 0:
#             return  # nothing to do

#         # State transition
#         F = np.array([
#             [1, 0, dt, 0],
#             [0, 1, 0, dt],
#             [0, 0, 1, 0 ],
#             [0, 0, 0, 1 ],
#         ], dtype=float)

#         # Process noise
#         q_pos = process_noise_pos
#         q_vel = process_noise_vel
#         Q = np.diag([q_pos, q_pos, q_vel, q_vel])

#         self.x = F @ self.x
#         self.P = F @ self.P @ F.T + Q
#         self.last_time = t_now

#     def update(self, z, t_now, row_idx, meas_noise_pos=0.5):
#         # Measurement matrix: we observe position only
#         H = np.array([
#             [1, 0, 0, 0],
#             [0, 1, 0, 0],
#         ], dtype=float)

#         R = np.eye(2) * meas_noise_pos

#         z = np.asarray(z, dtype=float)
#         y = z - H @ self.x                      # innovation
#         S = H @ self.P @ H.T + R
#         K = self.P @ H.T @ np.linalg.inv(S)     # Kalman gain

#         self.x = self.x + K @ y
#         I = np.eye(4)
#         self.P = (I - K @ H) @ self.P

#         self.last_time = t_now
#         self.hits += 1
#         self.row_indices.append(row_idx)


# def build_tracklets_from_detections(
#     df,
#     v_max=10.0,                  # m/s, ~36 km/h
#     max_gap_seconds=0.3,         # we don't allow gaps larger than this (but we kill earlier anyway)
#     confirm_min_duration=0.5,    # track must live at least this long to be "confirmed"
#     min_tracklet_duration=0.5    # keep only tracklets at least this long
# ):
#     """
#     Build conservative tracklets from detections in df.
#     df must have columns: 'timestamp', 'centroid_x', 'centroid_y'
    
#     Returns a copy of df with a new column 'tracklet_id' (int, -1 for unassigned).
#     """

#     # Work on a sorted copy, but remember original indices
#     df_sorted = df.sort_values("timestamp").reset_index(drop=False)
#     original_index = df_sorted["index"].to_numpy()

#     n = len(df_sorted)
#     tracklet_ids_sorted = np.full(n, fill_value=-1, dtype=int)

#     # Group detections by timestamp
#     # Each group key is timestamp, value is array of row indices in df_sorted
#     grouped = df_sorted.groupby("timestamp").indices
#     sorted_times = np.array(sorted(grouped.keys()))

#     active_tracks = []
#     next_internal_id = 0      # internal track counter
#     next_tracklet_id = 0      # external tracklet ID we write to df

#     def finalize_track(track):
#         nonlocal next_tracklet_id
        
#         duration = track.last_time - track.first_time
#         if (not track.confirmed) or (duration < min_tracklet_duration):
#             # discard weak/short tracks
#             return
        
#         # Assign a new global tracklet_id to all rows in this track
#         tracklet_ids_sorted[track.row_indices] = next_tracklet_id
#         next_tracklet_id += 1

#     for t in sorted_times:
#         row_idxs = grouped[t]
#         det_x = df_sorted.loc[row_idxs, "centroid_x"].to_numpy()
#         det_y = df_sorted.loc[row_idxs, "centroid_y"].to_numpy()
#         detections = np.stack([det_x, det_y], axis=1)  # shape (M, 2)

#         # 1) Predict all active tracks forward to time t,
#         #    and drop tracks that have grown too old.
#         still_active = []
#         for track in active_tracks:
#             dt = t - track.last_time
#             if dt > max_gap_seconds:
#                 # end of tracklet
#                 finalize_track(track)
#             else:
#                 track.predict(t)
#                 still_active.append(track)
#         active_tracks = still_active

#         # 2) Associate tracks with detections (greedy nearest-neighbor with gating)
#         num_tracks = len(active_tracks)
#         num_dets = len(detections)

#         unmatched_track_indices = set(range(num_tracks))
#         unmatched_det_indices = set(range(num_dets))

#         matches = []

#         if num_tracks > 0 and num_dets > 0:
#             # Build distance matrix between predicted track positions and detections
#             track_positions = np.array([[tr.x[0], tr.x[1]] for tr in active_tracks])
#             # (T, D) pairwise distances
#             diff = track_positions[:, None, :] - detections[None, :, :]
#             dists = np.linalg.norm(diff, axis=2)

#             # For each track, compute the max allowed distance based on v_max and dt
#             dt_array = np.array([t - tr.last_time for tr in active_tracks])
#             # avoid dt <= 0 just in case
#             dt_array = np.maximum(dt_array, 1e-6)
#             max_dist_for_track = v_max * dt_array   # shape (T,)

#             # Gating mask: valid if d <= v_max * dt
#             gate = dists <= max_dist_for_track[:, None]

#             # Set invalid pairs to a very large distance
#             LARGE = 1e9
#             dists_masked = dists.copy()
#             dists_masked[~gate] = LARGE

#             # Greedy: pick smallest distance, then remove that track and detection, repeat
#             while True:
#                 min_idx = np.argmin(dists_masked)
#                 i, j = divmod(min_idx, num_dets)
#                 if dists_masked[i, j] >= LARGE:
#                     break  # no more valid pairs

#                 # Accept this match
#                 matches.append((i, j))
#                 unmatched_track_indices.discard(i)
#                 unmatched_det_indices.discard(j)

#                 # Remove this row and column from further consideration
#                 dists_masked[i, :] = LARGE
#                 dists_masked[:, j] = LARGE

#         # 3) Update matched tracks
#         for ti, di in matches:
#             track = active_tracks[ti]
#             det_idx = row_idxs[di]
#             z = detections[di]
#             track.update(z, t_now=t, row_idx=det_idx)

#             # Check if this track should now be confirmed
#             duration = track.last_time - track.first_time
#             if (not track.confirmed) and (duration >= confirm_min_duration):
#                 track.confirmed = True

#         # 4) Finalize unmatched tracks (we don't allow them to survive unmatched)
#         tracks_to_keep = []
#         for idx, track in enumerate(active_tracks):
#             if idx in unmatched_track_indices:
#                 # finalize and discard
#                 finalize_track(track)
#             else:
#                 tracks_to_keep.append(track)
#         active_tracks = tracks_to_keep

#         # 5) Create new tentative tracks for unmatched detections
#         for di in unmatched_det_indices:
#             det_idx = row_idxs[di]
#             x0 = detections[di]
#             new_track = Track(x0=x0, t0=t, row_idx=det_idx, track_id=next_internal_id)
#             next_internal_id += 1
#             active_tracks.append(new_track)

#     # After last time step, finalize any remaining tracks
#     for track in active_tracks:
#         finalize_track(track)

#     # Map tracklet IDs back to original df order
#     out = df.copy()
#     out["tracklet_id"] = -1
#     out.loc[original_index, "tracklet_id"] = tracklet_ids_sorted

#     return out

In [ ]:
# tracklet_df = build_tracklets_from_detections(
#     parsed_df,
#     v_max=10.0,               # ~36 km/h
#     max_gap_seconds=0.5,
#     confirm_min_duration=0.3,
#     min_tracklet_duration=0.3,
# )

# tracklet_df["tracklet_id"].value_counts().head()

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

class Track_og:
    def __init__(self, x0, t0, row_idx, track_id):
        self.id = track_id
        
        # State: [x, y, vx, vy]
        self.x = np.array([x0[0], x0[1], 0.0, 0.0], dtype=float)
        
        # KF Initialization
        # We start with high uncertainty for velocity
        self.P = np.diag([1.0, 1.0, 10.0, 10.0]) 

        self.state_time = t0       
        self.last_meas_time = t0   
        self.confirmed = False
        self.row_indices = [row_idx]

    @property
    def speed(self):
        return np.hypot(self.x[2], self.x[3])

    @property
    def velocity_dir(self):
        # Returns normalized velocity vector (vx, vy)
        s = self.speed
        if s < 0.1: # Avoid division by zero
            return np.array([0.0, 0.0])
        return np.array([self.x[2]/s, self.x[3]/s])

    def predict(self, t_now):
        dt = t_now - self.state_time
        if dt <= 0: return

        F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0 ],
            [0, 0, 0, 1 ],
        ], dtype=float)

        # Process Noise
        # Higher q allows for faster changes in motion
        q_pos = 1.0 * (dt**2)
        q_vel = 2.0 * dt
        Q = np.diag([q_pos, q_pos, q_vel, q_vel])

        self.x = F @ self.x
        self.P = F @ self.P @ F.T + Q
        self.state_time = t_now

    def update(self, z, t_now, row_idx, meas_noise_pos=0.5):
        H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
        R = np.eye(2) * meas_noise_pos
        
        z = np.asarray(z, dtype=float)
        y = z - H @ self.x
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.inv(S)

        self.x = self.x + K @ y
        I = np.eye(4)
        self.P = (I - K @ H) @ self.P

        self.last_meas_time = t_now
        self.row_indices.append(row_idx)


def calculate_directional_costs_og(tracks, detections, angle_weight=2.0, speed_threshold=1.0):
    num_tracks = len(tracks)
    num_dets = len(detections)
    
    track_pos = np.array([[tr.x[0], tr.x[1]] for tr in tracks])  # (T, 2)
    det_pos = detections  # (D, 2)
    
    diff = det_pos[None, :, :] - track_pos[:, None, :]  # (T, D, 2)
    dists = np.linalg.norm(diff, axis=2)  # (T, D)

    track_vels = np.array([tr.velocity_dir for tr in tracks])  # (T, 2)
    track_speeds = np.array([tr.speed for tr in tracks])       # (T,)

    diff_norm = diff / (dists[:, :, None] + 1e-6)  # (T, D, 2)

    # Cosine similarity between velocity and displacement
    cos_sim = np.einsum('ti,tdi->td', track_vels, diff_norm)  # (T, D)

    angle_penalty_factor = (1.0 - cos_sim)  # (T, D), maps [-1,1] -> [2,0]

    multipliers = np.ones((num_tracks, num_dets))
    
    # 1D mask for moving tracks
    moving_rows = track_speeds > speed_threshold  # (T,)

    # Apply angle penalty only for moving tracks
    multipliers[moving_rows, :] += angle_weight * angle_penalty_factor[moving_rows, :]

    cost_matrix = dists * multipliers
    return cost_matrix, dists

def build_tracklets_hungarian_og(
    df,
    v_max=15.0,                  
    max_gap_seconds=0.2,         
    min_tracklet_duration=0.3,
    direction_weight=1.5,   # How much we hate turning (0.0 = circle, 2.0 = strong cone)
    min_speed_for_dir=0.7   # m/s. Below this, we assume circular motion.
):
    df_sorted = df.sort_values("timestamp").reset_index(drop=False)
    original_index = df_sorted["index"].to_numpy()
    
    tracklet_ids_sorted = np.full(len(df_sorted), fill_value=-1, dtype=int)
    grouped = df_sorted.groupby("timestamp").indices
    sorted_times = np.array(sorted(grouped.keys()))

    active_tracks = []
    next_internal_id = 0
    next_tracklet_id = 0

    def finalize_track(track):
        nonlocal next_tracklet_id
        start_t = df_sorted.loc[track.row_indices[0], "timestamp"]
        end_t = track.last_meas_time
        if (track.confirmed) and ((end_t - start_t) >= min_tracklet_duration):
            tracklet_ids_sorted[track.row_indices] = next_tracklet_id
            next_tracklet_id += 1

    for t in sorted_times:
        row_idxs = grouped[t]
        detections = df_sorted.loc[row_idxs, ["centroid_x", "centroid_y"]].to_numpy()

        # 1) Predict
        still_active = []
        for track in active_tracks:
            if (t - track.last_meas_time) > max_gap_seconds:
                finalize_track(track)
            else:
                track.predict(t)
                still_active.append(track)
        active_tracks = still_active

        # 2) Association (Hungarian + Directional Cost)
        num_tracks = len(active_tracks)
        num_dets = len(detections)
        
        matches = []
        unmatched_track_indices = set(range(num_tracks))
        unmatched_det_indices = set(range(num_dets))

        if num_tracks > 0 and num_dets > 0:
            # A. Calculate Direction-Aware Cost Matrix
            cost_matrix, raw_dists = calculate_directional_costs_og(
                active_tracks, 
                detections, 
                angle_weight=direction_weight,
                speed_threshold=min_speed_for_dir
            )
            
            # B. Solve Global Assignment
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            
            # C. Gate Validation (Hard Gate on Distance Only)
            # We still need a hard gate to prevent matching things 50 meters away
            # just because they were the "best option left".
            for r, c in zip(row_ind, col_ind):
                track = active_tracks[r]
                
                # Check pure physical distance capability
                dt = t - track.last_meas_time
                max_dist = v_max * np.maximum(dt, 0.05)
                
                # We check the RAW distance, not the penalized cost
                if raw_dists[r, c] <= max_dist:
                    matches.append((r, c))
                    unmatched_track_indices.discard(r)
                    unmatched_det_indices.discard(c)

        # 3) Update
        for ti, di in matches:
            track = active_tracks[ti]
            track.update(detections[di], t, row_idxs[di])
            
            # Simple confirmation logic
            if not track.confirmed and len(track.row_indices) >= 3:
                track.confirmed = True

        # 4) New Tracks
        for di in unmatched_det_indices:
            new_track = Track_og(detections[di], t, row_idxs[di], next_internal_id)
            next_internal_id += 1
            active_tracks.append(new_track)

    for track in active_tracks:
        finalize_track(track)

    out = df.copy()
    out["tracklet_id"] = -1
    out.loc[original_index, "tracklet_id"] = tracklet_ids_sorted
    return out


In [ ]:
tracklet_df_og = build_tracklets_hungarian_og(
    parsed_df,
    v_max=15.0,
    direction_weight=1.5,   # High penalty for incorrect direction
    min_speed_for_dir=0.7   # Only apply cones if moving > 0.7 m/s (~2.5 km/h)
)

In [ ]:
tracklet_df_og.info()

In [ ]:
show(tracklet_df_og)

In [ ]:
tracklet_df_og.groupby("tracklet_id").size()

In [ ]:
tracklet_df_og["tracklet_id"].value_counts().head()

In [ ]:

class Track:
    def __init__(self, x0, t0, row_idx, track_id):
        self.id = track_id
        
        # State: [x, y, vx, vy]
        self.x = np.array([x0[0], x0[1], 0.0, 0.0], dtype=float)
        
        # KF Initialization
        # We start with high uncertainty for velocity
        self.P = np.diag([1.0, 1.0, 10.0, 10.0]) 

        self.state_time = t0       
        self.last_meas_time = t0   
        self.confirmed = False
        self.row_indices = [row_idx]

    @property
    def speed(self):
        return np.hypot(self.x[2], self.x[3])

    @property
    def velocity_dir(self):
        # Returns normalized velocity vector (vx, vy)
        s = self.speed
        if s < 0.1: # Avoid division by zero
            return np.array([0.0, 0.0])
        return np.array([self.x[2]/s, self.x[3]/s])

    def predict(self, t_now):
        dt = t_now - self.state_time
        if dt <= 0: return

        F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0 ],
            [0, 0, 0, 1 ],
        ], dtype=float)

        # Process Noise
        # Higher q allows for faster changes in motion
        sigma_a = 3.0  # m/s^2 acceleration
        dt2 = dt*dt
        dt3 = dt2*dt
        dt4 = dt2*dt2
        q = sigma_a**2

        Q = q * np.array([
            [dt4/4, 0,     dt3/2, 0],
            [0,     dt4/4, 0,     dt3/2],
            [dt3/2, 0,     dt2,   0],
            [0,     dt3/2, 0,     dt2]
        ], dtype=float)

        self.x = F @ self.x
        self.P = F @ self.P @ F.T + Q
        self.state_time = t_now

    def update(self, z, t_now, row_idx, meas_noise_pos=0.25):
        H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
        R = np.eye(2) * meas_noise_pos
        
        z = np.asarray(z, dtype=float)
        y = z - H @ self.x
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.inv(S)

        self.x = self.x + K @ y
        I = np.eye(4)
        self.P = (I - K @ H) @ self.P

        self.last_meas_time = t_now
        self.row_indices.append(row_idx)


def calculate_directional_costs(tracks, detections, angle_weight=2.0, speed_threshold=1.0):
    num_tracks = len(tracks)
    num_dets = len(detections)
    
    track_pos = np.array([[tr.x[0], tr.x[1]] for tr in tracks])  # (T, 2)
    det_pos = detections  # (D, 2)
    
    diff = det_pos[None, :, :] - track_pos[:, None, :]  # (T, D, 2)
    dists = np.linalg.norm(diff, axis=2)  # (T, D)

    track_vels = np.array([tr.velocity_dir for tr in tracks])  # (T, 2)
    track_speeds = np.array([tr.speed for tr in tracks])       # (T,)

    diff_norm = diff / (dists[:, :, None] + 1e-6)  # (T, D, 2)

    # Cosine similarity between velocity and displacement
    cos_sim = np.einsum('ti,tdi->td', track_vels, diff_norm)  # (T, D)

    angle_penalty_factor = (1.0 - cos_sim)  # (T, D), maps [-1,1] -> [2,0]

    multipliers = np.ones((num_tracks, num_dets))
    
    # 1D mask for moving tracks
    moving_rows = track_speeds > speed_threshold  # (T,)

    # Apply angle penalty only for moving tracks
    multipliers[moving_rows, :] += angle_weight * angle_penalty_factor[moving_rows, :]

    cost_matrix = dists * multipliers
    return cost_matrix, dists

def build_tracklets_hungarian(
    df,
    v_max=15.0,
    a_max=5.0,
    max_gap_seconds=0.2,
    min_tracklet_duration=0.3,
    direction_weight=1.5,
    min_speed_for_dir=0.7,
    meas_noise_pos=0.25,       # measurement variance (m^2), same meaning as in update()
    gate_margin=0.5,
    vel_gate_min_hits=4,       # don’t trust KF speed until this many detections in the track
    use_kinematic_gate=True,
    use_mahalanobis_gate=True,
    maha_gamma=9.21            # chi-square threshold (2D): 9.21 ~ 99%
):
    df_sorted = df.sort_values("timestamp").reset_index(drop=False)
    original_index = df_sorted["index"].to_numpy()
    
    tracklet_ids_sorted = np.full(len(df_sorted), fill_value=-1, dtype=int)
    grouped = df_sorted.groupby("timestamp").indices
    sorted_times = np.array(sorted(grouped.keys()))

    active_tracks = []
    next_internal_id = 0
    next_tracklet_id = 0

    def finalize_track(track):
        nonlocal next_tracklet_id
        start_t = df_sorted.loc[track.row_indices[0], "timestamp"]
        end_t = track.last_meas_time
        if (track.confirmed) and ((end_t - start_t) >= min_tracklet_duration):
            tracklet_ids_sorted[track.row_indices] = next_tracklet_id
            next_tracklet_id += 1

    for t in sorted_times:
        row_idxs = grouped[t]
        detections = df_sorted.loc[row_idxs, ["centroid_x", "centroid_y"]].to_numpy()

        # 1) Predict
        still_active = []
        for track in active_tracks:
            if (t - track.last_meas_time) > max_gap_seconds:
                finalize_track(track)
            else:
                track.predict(t)
                still_active.append(track)
        active_tracks = still_active

        # 2) Association (Hungarian + Directional Cost)
        num_tracks = len(active_tracks)
        num_dets = len(detections)
        
        matches = []
        unmatched_track_indices = set(range(num_tracks))
        unmatched_det_indices = set(range(num_dets))

        if num_tracks > 0 and num_dets > 0:
            # A. Calculate Direction-Aware Cost Matrix
            cost_matrix, raw_dists = calculate_directional_costs(
                active_tracks, 
                detections, 
                angle_weight=direction_weight,
                speed_threshold=min_speed_for_dir
            )

            BIG = 1e9
            
            # We will build a final validity mask valid[T,D].
            # Start with everything valid, and AND-in each enabled gate.
            valid = np.ones((num_tracks, num_dets), dtype=bool)

            if use_kinematic_gate:
                dt_vec = np.maximum(
                    t - np.array([tr.last_meas_time for tr in active_tracks], dtype=float),
                    0.05
                )  # (T,)

                max_dist_vec = np.empty(num_tracks, dtype=float)
                for i, tr in enumerate(active_tracks):
                    dt_i = dt_vec[i]
                    if len(tr.row_indices) < vel_gate_min_hits:
                        # no reliable velocity yet -> permissive
                        max_dist_vec[i] = v_max * dt_i
                    else:
                        v = tr.speed
                        max_dist_vec[i] = v * dt_i + 0.5 * a_max * (dt_i ** 2) + gate_margin

                valid &= (raw_dists <= max_dist_vec[:, None])

            if use_mahalanobis_gate:
                # diff is the residual y = z - Hx, but in 2D position space
                track_pos = np.array([[tr.x[0], tr.x[1]] for tr in active_tracks], dtype=float)  # (T,2)
                det_pos = detections.astype(float)                                                # (D,2)
                diff = det_pos[None, :, :] - track_pos[:, None, :]                                # (T,D,2)

                H = np.array([[1,0,0,0],[0,1,0,0]], dtype=float)
                R = np.eye(2, dtype=float) * meas_noise_pos

                maha2 = np.empty((num_tracks, num_dets), dtype=float)

                for i, tr in enumerate(active_tracks):
                    S = H @ tr.P @ H.T + R             # (2,2)
                    S_inv = np.linalg.inv(S)           # (2,2)
                    # For each detection j: diff[i,j]^T S_inv diff[i,j]
                    maha2[i, :] = np.einsum('dj,jk,dk->d', diff[i], S_inv, diff[i])

                valid &= (maha2 <= maha_gamma)

            cost_matrix[~valid] = BIG

            # B. Solve Global Assignment (Hungarian)
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            
            # C. Hard gating using the same logic as before
            for r, c in zip(row_ind, col_ind):
                if valid[r, c]:
                    matches.append((r, c))
                    unmatched_track_indices.discard(r)
                    unmatched_det_indices.discard(c)

        # 3) Update
        for ti, di in matches:
            track = active_tracks[ti]
            track.update(detections[di], t, row_idxs[di])
            
            # Simple confirmation logic
            if not track.confirmed and len(track.row_indices) >= 3:
                track.confirmed = True

        # 4) New Tracks
        for di in unmatched_det_indices:
            new_track = Track(detections[di], t, row_idxs[di], next_internal_id)
            next_internal_id += 1
            active_tracks.append(new_track)

    for track in active_tracks:
        finalize_track(track)

    out = df.copy()
    out["tracklet_id"] = -1
    out.loc[original_index, "tracklet_id"] = tracklet_ids_sorted
    return out


In [ ]:
tracklet_df = build_tracklets_hungarian(
    parsed_df,
    v_max=15.0,
    a_max=5.0,
    gate_margin=0.5,
    direction_weight=1.5,   # High penalty for incorrect direction
    min_speed_for_dir=0.7,   # Only apply cones if moving > 0.7 m/s (~2.5 km/h)
    use_kinematic_gate=False,
)

tracklet_df["tracklet_id"].value_counts().head()

In [ ]:
tracklet_df.info()

In [ ]:
show(tracklet_df)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

# -----------------------------
# Helpers: Union-Find
# -----------------------------
class UnionFind:
    def __init__(self, items):
        self.parent = {int(i): int(i) for i in items}
        self.rank = {int(i): 0 for i in items}

    def find(self, a):
        a = int(a)
        while self.parent[a] != a:
            self.parent[a] = self.parent[self.parent[a]]
            a = self.parent[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return ra
        # union by rank
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return ra


# -----------------------------
# Helpers: velocity + length per tracklet
# -----------------------------
def _estimate_velocity_end(tracklet_df, k=6):
    """Linear fit on last k points: x(t), y(t) -> vx, vy at end. Returns (vx, vy)."""
    if len(tracklet_df) < 2:
        return 0.0, 0.0
    tail = tracklet_df.sort_values("timestamp").tail(k)
    t = tail["timestamp"].to_numpy(float)
    x = tail["centroid_x"].to_numpy(float)
    y = tail["centroid_y"].to_numpy(float)

    # shift time for conditioning
    t0 = t[0]
    tt = t - t0
    if np.allclose(tt, 0):
        return 0.0, 0.0

    vx, _ = np.polyfit(tt, x, 1)
    vy, _ = np.polyfit(tt, y, 1)
    return float(vx), float(vy)

def _tracklet_path_length(tracklet_df):
    """Sum of Euclidean distances along the tracklet."""
    g = tracklet_df.sort_values("timestamp")
    dx = g["centroid_x"].diff().to_numpy(float)
    dy = g["centroid_y"].diff().to_numpy(float)
    d = np.sqrt(np.nan_to_num(dx)**2 + np.nan_to_num(dy)**2)
    return float(np.nansum(d))

def build_tracklet_summaries(df, vel_fit_k=6):
    """Summaries: start/end positions/times, end velocity estimate, path length."""
    rows = []
    for tid, g in df.groupby("tracklet_id"):
        tid = int(tid)
        if tid < 0:
            continue
        g = g.sort_values("timestamp")
        s = g.iloc[0]
        e = g.iloc[-1]
        vx, vy = _estimate_velocity_end(g, k=vel_fit_k)
        length = _tracklet_path_length(g)
        rows.append({
            "tracklet_id": tid,
            "t_start": float(s["timestamp"]),
            "x_start": float(s["centroid_x"]),
            "y_start": float(s["centroid_y"]),
            "t_end": float(e["timestamp"]),
            "x_end": float(e["centroid_x"]),
            "y_end": float(e["centroid_y"]),
            "vx_end": float(vx),
            "vy_end": float(vy),
            "len": float(length),
            "n": int(len(g)),
        })
    return pd.DataFrame(rows)


# -----------------------------
# KF core (constant velocity)
# -----------------------------
def kf_predict(x, P, dt, q_pos=1.0, q_vel=2.0):
    """
    Constant-velocity KF predict.
    Q is simplified diagonal, like your tracklet builder.
    """
    if dt <= 0:
        return x, P

    F = np.array([
        [1, 0, dt, 0],
        [0, 1, 0, dt],
        [0, 0, 1,  0],
        [0, 0, 0,  1],
    ], dtype=float)

    Q = np.diag([
        q_pos * (dt**2),
        q_pos * (dt**2),
        q_vel * dt,
        q_vel * dt,
    ]).astype(float)

    x2 = F @ x
    P2 = F @ P @ F.T + Q
    return x2, P2

def kf_update_pos(x, P, z, meas_var=0.25):
    """
    Position-only update with R = meas_var * I.
    meas_var is variance in meters^2.
    """
    H = np.array([[1, 0, 0, 0],
                  [0, 1, 0, 0]], dtype=float)
    R = np.eye(2, dtype=float) * float(meas_var)

    z = np.asarray(z, dtype=float).reshape(2,)
    y = z - (H @ x)
    S = H @ P @ H.T + R
    K = P @ H.T @ np.linalg.inv(S)

    x2 = x + K @ y
    I = np.eye(4, dtype=float)
    P2 = (I - K @ H) @ P
    return x2, P2, y, S


# -----------------------------
# Active hypothesis (stitched end)
# -----------------------------
class ActiveHyp:
    __slots__ = ("rep_tid", "x", "P", "t_end", "length")

    def __init__(self, rep_tid, x, P, t_end, length):
        self.rep_tid = int(rep_tid)  # representative tracklet id for union-find / component
        self.x = x.astype(float)
        self.P = P.astype(float)
        self.t_end = float(t_end)
        self.length = float(length)

    @property
    def speed(self):
        return float(np.hypot(self.x[2], self.x[3]))

    @property
    def vdir(self):
        s = self.speed
        if s < 1e-6:
            return np.array([0.0, 0.0])
        return np.array([self.x[2]/s, self.x[3]/s], dtype=float)


# -----------------------------
# Main: offline stitching
# -----------------------------
def stitch_tracklets_offline_kf_hungarian(
    df,
    max_T=2.5,                 # seconds, occlusion tolerance
    max_len=70.0,              # meters, typical junction length ~60 => stop at ~70
    meas_var=0.25,             # meters^2, measurement variance (0.5m std -> 0.25 var)
    init_pos_var=1.0,          # initial P position variance
    init_vel_var=9.0,          # initial P velocity variance
    q_pos=1.0,                 # process noise scale (position)
    q_vel=2.0,                 # process noise scale (velocity)
    chi2_gate=9.21,            # 2D chi-square gate (≈99% => 9.21, 95% => 5.99)
    lambda_dir=1.5,            # direction penalty weight (additive to d_M^2)
    min_speed_for_dir=0.7,     # below this, don't enforce direction
    vel_fit_k=6,
    min_tracklet_points_for_stitch=3,  # avoid stitching from ultra-short junk
):
    """
    Returns:
      df_out with column 'stitched_id'
      links dataframe with accepted stitches
    """
    df_out = df.copy()
    summaries = build_tracklet_summaries(df_out, vel_fit_k=vel_fit_k)
    if summaries.empty:
        df_out["stitched_id"] = df_out["tracklet_id"].astype(int)
        return df_out, pd.DataFrame(columns=["from_rep", "to_tid", "dt", "d2", "dir_pen"])

    # Union-Find over tracklet_ids (final stitched components)
    all_tids = summaries["tracklet_id"].to_numpy(int)
    uf = UnionFind(all_tids)

    # Sort starts chronologically; batch by identical t_start (works well offline)
    summaries = summaries.sort_values("t_start").reset_index(drop=True)
    by_t = summaries.groupby("t_start", sort=True)

    active = []
    used_as_successor = set()

    links = []

    BIG = 1e12

    for t_start, batch in by_t:
        t_start = float(t_start)

        # 1) Expire / terminate active hyps by TTL or max_len
        still = []
        for hyp in active:
            if (t_start - hyp.t_end) > max_T:
                continue
            if hyp.length > max_len:
                continue
            still.append(hyp)
        active = still

        # 2) Candidate starts in this time batch that are not already consumed as successor
        batch = batch[~batch["tracklet_id"].isin(used_as_successor)].copy()
        if batch.empty:
            continue

        # If no active hyps, everything in batch becomes new hypothesis
        if len(active) == 0:
            for _, row in batch.iterrows():
                tid = int(row["tracklet_id"])
                x = np.array([row["x_end"], row["y_end"], row["vx_end"], row["vy_end"]], dtype=float)
                P = np.diag([init_pos_var, init_pos_var, init_vel_var, init_vel_var]).astype(float)
                hyp = ActiveHyp(rep_tid=tid, x=x, P=P, t_end=row["t_end"], length=row["len"])
                active.append(hyp)
            continue

        # 3) Build cost matrix: active ends -> batch starts
        starts = batch.reset_index(drop=True)
        A = len(active)
        B = len(starts)

        cost = np.full((A, B), BIG, dtype=float)
        d2_mat = np.full((A, B), BIG, dtype=float)
        dir_pen_mat = np.zeros((A, B), dtype=float)

        # Start measurements
        z_starts = starts[["x_start", "y_start"]].to_numpy(float)
        t_starts = starts["t_start"].to_numpy(float)

        for i, hyp in enumerate(active):
            # don’t stitch from very short components if you want (optional heuristic)
            # Here we gate based on the candidate successor's own point count; you can change this.
            # We'll use min_tracklet_points_for_stitch below per-column.

            dt = t_starts - hyp.t_end
            feasible = (dt > 0) & (dt <= max_T)
            if not np.any(feasible):
                continue

            for j in np.where(feasible)[0]:
                # Optional: avoid stitching into very short tracklets (velocity/direction unreliable)
                if int(starts.loc[j, "n"]) < min_tracklet_points_for_stitch:
                    continue

                dt_ij = float(dt[j])

                # Predict to the start time
                x_pred, P_pred = kf_predict(hyp.x, hyp.P, dt_ij, q_pos=q_pos, q_vel=q_vel)

                # Mahalanobis gate via update innovation
                z = z_starts[j]
                _, _, innov, S = kf_update_pos(x_pred, P_pred, z, meas_var=meas_var)
                d2 = float(innov.T @ np.linalg.inv(S) @ innov)

                if d2 > chi2_gate:
                    continue

                # Direction continuity penalty (soft)
                dir_pen = 0.0
                if hyp.speed > min_speed_for_dir:
                    disp = (z - x_pred[:2])
                    disp_norm = np.linalg.norm(disp)
                    if disp_norm > 1e-6:
                        disp_dir = disp / disp_norm
                        cos_sim = float(disp_dir @ hyp.vdir)  # [-1..1]
                        dir_pen = (1.0 - cos_sim)            # [0..2]
                c = d2 + lambda_dir * dir_pen

                cost[i, j] = c
                d2_mat[i, j] = d2
                dir_pen_mat[i, j] = dir_pen

        # 4) Hungarian assignment on this batch
        row_ind, col_ind = linear_sum_assignment(cost)

        matched_hyps = set()
        matched_starts = set()

        for r, c in zip(row_ind, col_ind):
            if cost[r, c] >= BIG/10:
                continue

            hyp = active[r]
            to_tid = int(starts.loc[c, "tracklet_id"])

            # Accept (already chi2-gated)
            matched_hyps.add(r)
            matched_starts.add(c)

            # Merge components: hyp.rep_tid -> to_tid
            rep = uf.union(hyp.rep_tid, to_tid)
            hyp.rep_tid = rep

            # KF: predict to start time and update with start measurement
            dt1 = float(starts.loc[c, "t_start"] - hyp.t_end)
            hyp.x, hyp.P = kf_predict(hyp.x, hyp.P, dt1, q_pos=q_pos, q_vel=q_vel)
            hyp.x, hyp.P, _, _ = kf_update_pos(hyp.x, hyp.P, z_starts[c], meas_var=meas_var)

            # KF: then incorporate end measurement of the successor tracklet
            t_end_succ = float(starts.loc[c, "t_end"])
            z_end = np.array([starts.loc[c, "x_end"], starts.loc[c, "y_end"]], dtype=float)
            dt2 = float(t_end_succ - float(starts.loc[c, "t_start"]))
            if dt2 > 0:
                hyp.x, hyp.P = kf_predict(hyp.x, hyp.P, dt2, q_pos=q_pos, q_vel=q_vel)
                hyp.x, hyp.P, _, _ = kf_update_pos(hyp.x, hyp.P, z_end, meas_var=meas_var)

            # Update hyp end time
            prev_end_xy = hyp.x[:2].copy()
            hyp.t_end = t_end_succ

            # Length bookkeeping:
            # add the physical gap from predicted end-at-start (approx) + successor tracklet length
            gap = float(np.linalg.norm(z_starts[c] - prev_end_xy))
            hyp.length += gap + float(starts.loc[c, "len"])

            used_as_successor.add(to_tid)

            links.append({
                "from_rep": int(rep),
                "to_tid": to_tid,
                "dt": float(starts.loc[c, "t_start"] - (t_start - 0.0)),  # informational only
                "d2": float(d2_mat[r, c]),
                "dir_pen": float(dir_pen_mat[r, c]),
            })

        # 5) Any unmatched starts become new hypotheses
        for j in range(B):
            if j in matched_starts:
                continue
            tid = int(starts.loc[j, "tracklet_id"])
            x = np.array([starts.loc[j, "x_end"], starts.loc[j, "y_end"],
                          starts.loc[j, "vx_end"], starts.loc[j, "vy_end"]], dtype=float)
            P = np.diag([init_pos_var, init_pos_var, init_vel_var, init_vel_var]).astype(float)
            active.append(ActiveHyp(rep_tid=tid, x=x, P=P, t_end=float(starts.loc[j, "t_end"]), length=float(starts.loc[j, "len"])))

    # Produce stitched_id per tracklet_id
    stitched_map = {int(t): int(uf.find(t)) for t in all_tids}
    df_out["stitched_id"] = df_out["tracklet_id"].astype(int).map(stitched_map)

    links_df = pd.DataFrame(links)
    return df_out, links_df

In [ ]:
final_tracks, links_df = stitch_tracklets_offline_kf_hungarian(
    tracklet_df
)

In [ ]:
show(final_tracks)

In [ ]:
links_df.head()

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

class SuperTrack:
    """
    A 'SuperTrack' is a chain of stitched tracklets.
    It tracks the state at the TAIL (the end of the last stitched tracklet).
    """
    def __init__(self, tracklet_meta, super_id):
        self.super_id = super_id
        self.tracklet_ids = [tracklet_meta['id']]
        
        # We initialize our state using the TAIL of the first tracklet
        # State: [x, y, vx, vy] at the END of the tracklet
        self.x = np.array([
            tracklet_meta['end_x'], 
            tracklet_meta['end_y'], 
            tracklet_meta['end_vx'], 
            tracklet_meta['end_vy']
        ])
        
        self.t_tail = tracklet_meta['end_t']  # Time at the tail
        
        # Initial covariance
        # We are fairly confident in the tracklet's end state
        self.P = np.diag([0.5, 0.5, 1.0, 1.0]) 

    def predict_to(self, t_target, process_noise_scale=2.0):
        """
        Predicts state from self.t_tail to t_target.
        Returns (predicted_x, predicted_P) without modifying internal state.
        This allows us to compare against multiple candidates.
        """
        dt = t_target - self.t_tail
        
        # If dt is negative, this track ended AFTER the new one started. 
        # Impossible to stitch (temporal violation).
        if dt < 0:
            return None, None
            
        # F Matrix (Constant Velocity)
        F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0 ],
            [0, 0, 0, 1 ]
        ])
        
        # Q Matrix (Process Noise)
        # As you requested: Uncertainty grows with time.
        # We scale noise by dt so gaps create larger uncertainty clouds.
        q_pos = process_noise_scale * (dt**2)
        q_vel = process_noise_scale * dt
        Q = np.diag([q_pos, q_pos, q_vel, q_vel])
        
        x_pred = F @ self.x
        P_pred = F @ self.P @ F.T + Q
        
        return x_pred, P_pred

    def absorb(self, tracklet_meta):
        """
        Stitch a new tracklet to this SuperTrack.
        We jump our state to the END of the new tracklet.
        """
        self.tracklet_ids.append(tracklet_meta['id'])
        
        # We trust the tracklet builder's local velocity for the new segment
        # So we simply reset our state to the new tracklet's tail.
        # Alternatively, you could run a KF update step here, but hard-resetting 
        # is often more robust because the tracklet builder has 'seen' the real data.
        self.x = np.array([
            tracklet_meta['end_x'], 
            tracklet_meta['end_y'], 
            tracklet_meta['end_vx'], 
            tracklet_meta['end_vy']
        ])
        self.t_tail = tracklet_meta['end_t']
        
        # Reset covariance slightly, but keep some history if desired.
        # Resetting is safer to prevent covariance explosion from sticking.
        self.P = np.diag([0.5, 0.5, 1.0, 1.0]) 

def prepare_meta_tracklets(df):
    """
    Summarizes each tracklet into a single dictionary of properties.
    """
    meta_list = []
    # Group by tracklet ID
    for tid, group in df.groupby("tracklet_id"):
        group = group.sort_values("timestamp")
        
        start_row = group.iloc[0]
        end_row = group.iloc[-1]
        
        # Calculate robust velocity (Start to End)
        duration = end_row["timestamp"] - start_row["timestamp"]
        duration = max(duration, 0.05)
        
        vx = (end_row["centroid_x"] - start_row["centroid_x"]) / duration
        vy = (end_row["centroid_y"] - start_row["centroid_y"]) / duration
        
        meta = {
            'id': tid,
            'start_t': start_row["timestamp"],
            'end_t': end_row["timestamp"],
            'start_x': start_row["centroid_x"],
            'start_y': start_row["centroid_y"],
            'start_vx': vx,  # We use the average velocity as the "entrance" velocity
            'start_vy': vy,
            'end_x': end_row["centroid_x"],
            'end_y': end_row["centroid_y"],
            'end_vx': vx,    # And as the "exit" velocity
            'end_vy': vy,
        }
        meta_list.append(meta)
        
    # Sort by START time (Sequential Processing)
    return sorted(meta_list, key=lambda x: x['start_t'])

def stitch_tracklets_sequential(
    df,
    max_gap_time=5.0,        # Max time an object can be occluded
    max_dist_gating=10.0,    # Max reasonable jump distance even with high covariance
    direction_weight=2.0
):
    # 1. Prepare Data
    tracklets = prepare_meta_tracklets(df)
    
    active_super_tracks = []
    finished_super_tracks = []
    
    next_super_id = 0
    
    # 2. Sequential Loop
    for new_trk in tracklets:
        t_now = new_trk['start_t']
        
        # A. Clean up old tracks (Time Gating)
        # If a SuperTrack's tail is too old, it's finished.
        active = []
        for st in active_super_tracks:
            if (t_now - st.t_tail) > max_gap_time:
                finished_super_tracks.append(st)
            else:
                active.append(st)
        active_super_tracks = active
        
        # B. Associate
        num_active = len(active_super_tracks)
        
        if num_active == 0:
            # No active tracks, start a new one
            new_st = SuperTrack(new_trk, next_super_id)
            next_super_id += 1
            active_super_tracks.append(new_st)
            continue
            
        # Build Cost Matrix (Rows=ActiveTracks, Cols=NewTracklet(1))
        # We are matching 1 new tracklet against N active tracks.
        # This is a 1-to-N matching, simpler than full M-to-N.
        
        best_cost = float('inf')
        best_idx = -1
        
        costs = []
        
        # New tracklet state vector (Position + Velocity)
        z_new = np.array([new_trk['start_x'], new_trk['start_y'], new_trk['start_vx'], new_trk['start_vy']])
        
        for i, st in enumerate(active_super_tracks):
            # Predict SuperTrack to the START time of the new tracklet
            pred_x, pred_P = st.predict_to(t_now)
            
            if pred_x is None: # Temporal violation
                costs.append(float('inf'))
                continue
                
            # --- 1. Mahalanobis Distance (Approx Pimated) ---
            # Innovation (Difference between Pred and New)
            residual = z_new - pred_x # [dx, dy, dvx, dvy]
            
            # We focus mainly on Position Residual for gating
            pos_resid = residual[:2]
            dist = np.linalg.norm(pos_resid)
            
            # Use P matrix to scale distance (Growing Uncertainty)
            # Simple approximation: dist / sqrt(trace(P_pos))
            uncertainty_factor = np.sqrt(pred_P[0,0] + pred_P[1,1])
            mahala_cost = dist / uncertainty_factor
            
            # --- 2. Directional Similarity ---
            # Dot product of SuperTrack velocity vs NewTracklet velocity
            # Normalize vectors
            v_st = pred_x[2:4]
            v_new = z_new[2:4]
            
            norm_st = np.linalg.norm(v_st)
            norm_new = np.linalg.norm(v_new)
            
            dir_penalty = 0
            if norm_st > 1.0 and norm_new > 1.0: # Only if moving
                cos_sim = np.dot(v_st, v_new) / (norm_st * norm_new)
                # cos_sim: 1.0 (Same dir), -1.0 (Opposite)
                # Penalty: 0.0 to 2.0
                dir_penalty = (1.0 - cos_sim) * direction_weight
            
            total_cost = mahala_cost + dir_penalty
            
            # --- 3. Hard Gating ---
            # Even if uncertainty is huge, we don't want to jump 100 meters
            if dist > max_dist_gating:
                total_cost = float('inf')
                
            costs.append(total_cost)
            
        # Find best match
        costs = np.array(costs)
        if len(costs) > 0:
            min_c = np.min(costs)
            if min_c < 4.0: # Threshold for valid match (Tunable)
                best_idx = np.argmin(costs)
        
        # C. Update
        if best_idx != -1:
            # Match found! Extend the SuperTrack
            active_super_tracks[best_idx].absorb(new_trk)
        else:
            # No match, start new SuperTrack
            new_st = SuperTrack(new_trk, next_super_id)
            next_super_id += 1
            active_super_tracks.append(new_st)
            
    # Finish remaining
    finished_super_tracks.extend(active_super_tracks)
    
    # 3. Map IDs back to DataFrame
    # Build a lookup dict: {tracklet_id: super_id}
    mapping = {}
    for st in finished_super_tracks:
        for tid in st.tracklet_ids:
            mapping[tid] = st.super_id
            
    out_df = df.copy()
    out_df['global_id'] = out_df['tracklet_id'].map(mapping)
    return out_df

# Usage
final_df = stitch_tracklets_sequential(tracklet_df, max_gap_time=3.0)

In [ ]:
show(final_df)

In [ ]:
SENSOR_LAT = 60.197547
SENSOR_LON = 24.907931

def lidar_xy_to_latlon(x, y, sensor_lat=SENSOR_LAT, sensor_lon=SENSOR_LON, rotation_deg=0.0):
    """
    Convert local LiDAR coordinates (meters) to WGS84 lat/lon using a simple
    flat-earth approximation around the sensor location.
    
    rotation_deg: rotation of LiDAR X axis relative to East (positive CCW).
                  0°  => x=east,  y=north
                  170° => roughly facing south (you can experiment with this)
    """
    # 1) rotate in the local ENU plane
    theta = np.deg2rad(rotation_deg)
    xr = x * np.cos(theta) - y * np.sin(theta)
    yr = x * np.sin(theta) + y * np.cos(theta)

    # 2) convert meters to lat/lon offsets
    R = 6378137.0  # Earth radius [m]
    dlat = (yr / R) * (180.0 / np.pi)
    dlon = (xr / (R * np.cos(np.deg2rad(sensor_lat)))) * (180.0 / np.pi)

    lat = sensor_lat + dlat
    lon = sensor_lon + dlon
    return lat, lon

In [ ]:
def most_common(s: pd.Series):
    mode = s.mode()
    return mode.iloc[0] if len(mode) else None

def make_line(points):
    pts = [(p.x, p.y) for p in points if p.is_valid]
    return LineString(pts) if len(pts) > 1 else None

def build_tracklet_geodata(tracklet_df: pd.DataFrame) -> gpd.GeoDataFrame:
    # Keep only assigned tracklets
    df = tracklet_df[tracklet_df["tracklet_id"] >= 0].copy()
    if df.empty:
        raise ValueError("No tracklets to visualize (tracklet_id < 0 only).")

    # Sort by tracklet + time
    df = df.sort_values(["tracklet_id", "timestamp"]).copy()

    # Convert timestamp to datetime (seconds!)
    df["timestamp_dt"] = pd.to_datetime(df["timestamp"], unit="s")

    # Geometry in LiDAR local coordinates
    df["center"] = df.apply(lambda r: Point(r["centroid_x"], r["centroid_y"]), axis=1)
    gdf_tracks = gpd.GeoDataFrame(df, geometry="center", crs=None)

    # Build one line per tracklet_id
    tracks = (
        gdf_tracks.groupby("tracklet_id")
        .agg(
            start_timestamp=("timestamp","min"),
            end_timestamp=("timestamp","max"),
            start_timestamp_dt=("timestamp_dt","min"),
            end_timestamp_dt=("timestamp_dt","max"),
            point_count=("center","count"),
            predicted_class=("predicted_class", most_common) if "predicted_class" in gdf_tracks.columns else ("tracklet_id", "count"),
            geometry=("center", make_line),
        )
        .reset_index()
        .rename(columns={"tracklet_id": "id"})  # for easier reuse of your patterns
    )

    # Handle degenerate tracks (only one point → no LineString)
    tracks_ok = tracks[tracks.geometry.notna()].copy()
    broken = tracks[tracks.geometry.isna()].copy()

    if not broken.empty:
        broken = broken.merge(
            gdf_tracks[["tracklet_id", "center"]],
            left_on="id",
            right_on="tracklet_id",
            how="left"
        )
        broken["geometry"] = broken["center"]
        broken = broken.drop(columns=["center", "tracklet_id"])

    tracks_ok["track_type"] = "TRACK"
    if not broken.empty:
        broken["track_type"] = "BROKEN_POINT"
        tracks_all = pd.concat([tracks_ok, broken], ignore_index=True)
    else:
        tracks_all = tracks_ok.copy()
        tracks_all["track_type"] = "TRACK"

    tracks_all = gpd.GeoDataFrame(tracks_all, geometry="geometry", crs=None)

    # Simple color palette
    PALETTE = ["red", "blue", "green", "orange", "purple"]
    tracks_all["color"] = [PALETTE[i % len(PALETTE)] for i in range(len(tracks_all))]

    return tracks_all

In [ ]:
def tracks_to_wgs84(tracks_all: gpd.GeoDataFrame, rotation_deg: float = 0.0) -> gpd.GeoDataFrame:
    rows = []
    for _, row in tracks_all.iterrows():
        geom = row.geometry
        if geom is None or not geom.is_valid:
            continue

        if geom.geom_type == "LineString":
            coords = []
            for x, y in geom.coords:
                lat, lon = lidar_xy_to_latlon(x, y, rotation_deg=rotation_deg)
                coords.append((lon, lat))  # GeoJSON / WGS84 order: (lon, lat)
            new_geom = LineString(coords)
        elif geom.geom_type == "Point":
            x, y = geom.x, geom.y
            lat, lon = lidar_xy_to_latlon(x, y, rotation_deg=rotation_deg)
            new_geom = Point(lon, lat)
        else:
            # ignore other geometry types for now
            continue

        new_row = row.copy()
        new_row.geometry = new_geom
        rows.append(new_row)

    gdf_wgs84 = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
    return gdf_wgs84

In [ ]:
def build_tracklet_timeline_map(
    tracklet_df: pd.DataFrame,
    output_html: str = "track_visualizations/tracklets_timeline_slider.html",
    rotation_deg: float = 0.0,
):
    tracks_all = build_tracklet_geodata(tracklet_df)
    tracks_all_wgs84 = tracks_to_wgs84(tracks_all, rotation_deg=rotation_deg)

    # Build GeoJSON manually (same pattern as your working code)
    features = []
    for _, row in tracks_all_wgs84.iterrows():
        geom = row.geometry
        if geom is None or not geom.is_valid:
            continue

        if geom.geom_type == "LineString":
            coordinates = [[x, y] for x, y in geom.coords]
        elif geom.geom_type == "Point":
            coordinates = [geom.x, geom.y]
        else:
            continue

        features.append({
            "type": "Feature",
            "geometry": {
                "type": geom.geom_type,
                "coordinates": coordinates,
            },
            "properties": {
                "id": row["id"],
                "track_type": row["track_type"],
                "predicted_class": row.get("predicted_class", None),
                "point_count": int(row["point_count"]),
                "start": float(row["start_timestamp"]),
                "end": float(row["end_timestamp"]),
                "start_timestamp_dt": row["start_timestamp_dt"].isoformat(),
                "end_timestamp_dt": row["end_timestamp_dt"].isoformat(),
                "color": row["color"],
            },
        })

    tracks_geojson = {
        "type": "FeatureCollection",
        "features": features,
    }

    # JS style function same as your pattern
    style_js = JsCode("""
        function (data) {
            return {
                color: data.properties.color,
                weight: 3,
                opacity: 0.8
            };
        }
    """)

    # Center map around average centroid
    centroids = tracks_all_wgs84.geometry.centroid
    center_lat = centroids.y.mean()
    center_lon = centroids.x.mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=18)

    # Reuse your Timeline / TimelineSlider pattern
    timeline = Timeline(
        data=tracks_geojson,
        style=style_js,
    ).add_to(m)

    tooltip = folium.GeoJsonTooltip(
        fields=["id", "track_type", "predicted_class", "point_count", "start_timestamp_dt", "end_timestamp_dt"],
        sticky=True,
    )
    tooltip.add_to(timeline)

    TimelineSlider(
        auto_play=False,
        show_ticks=True,
        enable_keyboard_controls=True,
        playback_duration=30000,
    ).add_timelines(timeline).add_to(m)

    m.save(output_html)
    print("Saved →", output_html)

In [ ]:
# After you have tracklet_df from build_tracklets_from_detections(...)
build_tracklet_timeline_map(
    tracklet_df,
    output_html="track_visualizations/tracklets_timeline_slider.html",
    rotation_deg=-45.0,  # or try 170.0 later
)

In [ ]:
# After you have tracklet_df from build_tracklets_from_detections(...)
build_tracklet_timeline_map(
    tracklet_df_og,
    output_html="track_visualizations/tracklets_timeline_slider_og.html",
    rotation_deg=-45.0,  # or try 170.0 later
)

In [ ]:
def build_tracklets_gdf(df: pd.DataFrame) -> gpd.GeoDataFrame:
    # keep only valid tracklets
    df = df[df["tracklet_id"] >= 0].copy()

    # sort so lines follow time
    df = df.sort_values(["tracklet_id", "timestamp"])

    # point geometry
    df["geometry"] = df.apply(
        lambda r: Point(r["centroid_x"], r["centroid_y"]), axis=1
    )
    gdf_points = gpd.GeoDataFrame(df, geometry="geometry", crs=None)

    # aggregate to tracklets
    tracks = (
        gdf_points.groupby("tracklet_id")
        .agg(
            start_timestamp=("timestamp", "min"),
            end_timestamp=("timestamp", "max"),
            point_count=("geometry", "count"),
            predicted_class=("predicted_class", lambda s: s.mode().iloc[0]),
            geometry=("geometry", lambda pts: LineString(pts.tolist()) if len(pts) > 1 else pts.iloc[0]),
            mean_speed=("speed", "mean")
        )
        .reset_index()
        .rename(columns={"tracklet_id": "id"})
    )

    tracks = gpd.GeoDataFrame(tracks, geometry="geometry", crs=None)

    # length in meters (local LiDAR frame)
    tracks["length_m"] = tracks.geometry.length

    return tracks

In [ ]:
geo_tracklets_df = build_tracklets_gdf(tracklet_df)
geo_tracklets_df_wgs84 = tracks_to_wgs84(geo_tracklets_df, -45)

In [ ]:
show(geo_tracklets_df_wgs84)

In [ ]:
geo_tracklets_df["length_m"] = geo_tracklets_df.geometry.length

fig_len_vs_points = px.scatter(
    geo_tracklets_df,
    x="point_count",
    y="length_m",
    color="predicted_class",
    opacity=0.3,
    labels={
        "point_count": "Number of points in track",
        "length_m": "Track length (m)",
        "predicted_class": "Predicted class",
    },
    title="Track length (m) vs. number of points",
)

fig_len_vs_points.update_layout(width=700, height=500)
fig_len_vs_points.show()

In [ ]:
fig_speed_vs_points = px.scatter(
    geo_tracklets_df,
    x="point_count",
    y="mean_speed",
    color="predicted_class",
    opacity=0.3,
    title="Mean speed vs. point count per track",
    labels={"mean_speed": "Mean speed (m/s)", "point_count": "Point count"},
)

fig_speed_vs_points.update_layout(width=700, height=500)
fig_speed_vs_points.show()

In [ ]:
classes = sorted(geo_tracklets_df["predicted_class"].dropna().unique())

for cls in classes:
    df_cls = geo_tracklets_df[geo_tracklets_df["predicted_class"] == cls]

    fig_pc = px.histogram(
        df_cls,
        x="point_count",
        nbins=40,
        title=f"Point count distribution - class: {cls}",
        labels={"point_count": "Number of points in track"},
    )
    fig_pc.update_layout(
        width=650,
        height=400,
        bargap=0.05,
    )
    fig_pc.show()

In [ ]:
for cls in classes:
    df_cls = geo_tracklets_df[geo_tracklets_df["predicted_class"] == cls]

    fig_len = px.histogram(
        df_cls,
        x="length_m",
        nbins=40,
        title=f"Track length (m) distribution - class: {cls}",
        labels={"length_m": "Track length (m)"},
    )
    fig_len.update_layout(
        width=650,
        height=400,
        bargap=0.05,
    )
    fig_len.show()